# Forex Trading Strategy Analysis

This notebook analyzes EUR/USD Forex trading data to identify profitable trading strategies based on various technical indicators and market conditions.

In [477]:
# Import required libraries
import pandas as pd
import numpy as np
from IPython.display import display, HTML
import sys
import importlib

# Force reload of utils module to prevent caching
if 'utils' in sys.modules:
    del sys.modules['utils']

# Enable automatic reloading of modules
%load_ext autoreload
%autoreload 2

# Import utility functions (will be freshly loaded)
import utils
from utils import (
    load_and_clean_data, 
    calculate_profitable_trades, 
    analyze_entry_timing, 
    display_profitable_strategies,
    calculate_rrr_stats,
    Strategy,
    create_strategy_library,
    evaluate_all_strategies,
    get_top_strategies
)

# Force reload utils to ensure latest version
importlib.reload(utils)

# Load the data
df = load_and_clean_data()

# Display profitable trades analysis
display(HTML("<h2>Profitable Trades</h2><p>What are simple trading ideas that are profitable?</p>"))
display(calculate_profitable_trades(df))
display(HTML("<p><b>Summary</b></p><ul><li>CH trades are out of question, they're not profitable.</li><li>3 pips extra is harming the strategy, while 1 pip can improve it slightly.</li><li>Overall any filtration is harmful.</ul>"))

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


,Data,1:1 RRR,1:2 RRR,1:3 RRR
0,With Extra 1 pip,42.9%,25.5%,18.4%
1,With Extra 2 pips,42.9%,27.6%,14.3%
2,Total,38.8%,24.5%,20.4%
3,With Extra 3 pips,36.7%,22.4%,13.3%
4,Just BOS Trades,25.5%,17.3%,15.3%
5,With EMA Direction,23.5%,14.3%,11.2%
6,With EMA + BOS,18.4%,12.2%,10.2%
7,Against EMA Direction,15.3%,10.2%,9.2%
8,Just CH Trades,13.3%,7.1%,5.1%
9,With EMA + CH,5.1%,2.0%,1.0%


In [478]:
# Display entry timing analysis using the imported function
display(HTML("<h2>When To Enter</h2><p>Following market structure, price taps order block. This is when the signal is received. Now - when to enter the trade?</p>"))
display(analyze_entry_timing(df))
display(HTML("<p><b>Summary</b></p><ul><li>Very interesting that here, my strategy with 1M confirmation candle is the worse.</li></ul>"))
display(HTML("<p><b>Open Questions</b></p><ul><li>That 5M Stop entry makes a lot of sense as all profitable strategies would have it, but in reality somehow it performs the worse. Why?</li><li>Why does 1M confirmation candle entry outperforms all other entries in the next setups analysis?</li></ul>"))

,Idea,Notation,Win Rate,With Extra,With Extra & 1:3 RRR
0,5M Stop,31W - 32L,49.2%,55.6%,28.6%
1,5M Breakout,30W - 35L,46.2%,52.3%,30.8%
2,5M Confirmation Candle,31W - 40L,43.7%,49.3%,28.2%
3,1M Confirmation Candle,38W - 60L,38.8%,42.9%,22.4%


In [479]:
# Initialize base strategy list
strategies = [
    Strategy(
        "Plain Strategy",
        lambda df: df,
        "Baseline: All trades without any filtering"
    )
]

# Add all strategies from the library
strategies.extend(create_strategy_library())

# Evaluate all strategies
strategy_results = evaluate_all_strategies(df, strategies)

# Display top performing strategies for each RRR
display(HTML("<h2>🏆 TOP Performing Strategies</h2>"))

rrr_configs = [
    ('1:1 RRR', '1:1'),
    ('1:2 RRR', '1:2'),
    ('1:3 RRR', '1:3'),
]

for rrr_column, rrr_label in rrr_configs:
    display(HTML(f"<h3>Best {rrr_label} Strategies</h3>"))
    
    # Get and display top 5 strategies
    top_df = get_top_strategies(strategy_results, rrr_column)
    top_df = top_df.rename(columns={'Strategy': f'Top {rrr_label} Strategies'})
    
    # Style the table for better readability
    styled_df = top_df.style.set_properties(
        subset=[f'Top {rrr_label} Strategies'], 
        **{'width': '300px', 'font-weight': 'bold'}
    ).set_properties(
        subset=['Outcome'],
        **{'color': 'green', 'font-weight': 'bold'}
    )
    
    display(styled_df)
    print()  # Add spacing

display(HTML("<p><b>Summary</b></p><ul><li>EMA + BOS is on 5th place. So anything above it should be tradeable</li><li>According this list, trading all BOS trades would simplify things a lot. Lets try that out!</ul>"))

,Top 1:1 Strategies,Entry,Trades,Wins,Losses,Win Rate,Edge,Outcome
0,BOS,1M CC,61,33,28,54.1%,4.1%,5R
1,EMA + Opposite Trade Direction,1M CC,38,21,17,55.3%,5.3%,4R
2,No News Events,1M CC,54,29,25,53.7%,3.7%,4R
3,Aggressive: SL >= 7 pips,1M CC,17,10,7,58.8%,8.8%,3R
4,Plain Strategy,1M CC,98,50,48,51.0%,1.0%,2R
5,With 30M Trend,1M CC,60,31,29,51.7%,1.7%,2R
6,News in 2+ Hours,1M CC,26,14,12,53.8%,3.8%,2R
7,Moderate Risk: SL 3-6 pips,1M CC,53,27,26,50.9%,0.9%,1R
8,BOS + Moderate SL,1M CC,35,18,17,51.4%,1.4%,1R
9,Aggressive: SL >= 7 pips,5M CC,17,9,8,52.9%,2.9%,1R


,Top 1:2 Strategies,Entry,Trades,Wins,Losses,Win Rate,Edge,Outcome
0,BOS,1M CC,61,24,37,39.3%,6.0%,11R
1,BOS,5M Stop,61,24,37,39.3%,6.0%,11R
2,News in 2+ Hours,1M CC,26,12,14,46.2%,12.9%,10R
3,With 30M Trend,5M Breakout,60,23,37,38.3%,5.0%,9R
4,BOS,5M CC,61,23,38,37.7%,4.4%,8R
5,BOS,5M Breakout,61,23,38,37.7%,4.4%,8R
6,News in 2+ Hours,5M CC,26,11,15,42.3%,9.0%,7R
7,News in 2+ Hours,5M Stop,26,11,15,42.3%,9.0%,7R
8,EMA + Opposite Trade Direction,5M Breakout,38,15,23,39.5%,6.2%,7R
9,News in 2+ Hours,5M Breakout,26,11,15,42.3%,9.0%,7R


,Top 1:3 Strategies,Entry,Trades,Wins,Losses,Win Rate,Edge,Outcome
0,BOS,1M CC,61,22,39,36.1%,11.1%,27R
1,BOS,5M Breakout,61,20,41,32.8%,7.8%,19R
2,With 30M Trend,1M CC,60,19,41,31.7%,6.7%,16R
3,BOS,5M CC,61,19,42,31.1%,6.1%,15R
4,Plain Strategy,1M CC,98,28,70,28.6%,3.6%,14R
5,EMA + Opposite Trade Direction,1M CC,38,13,25,34.2%,9.2%,14R
6,News in 2+ Hours,5M CC,26,10,16,38.5%,13.5%,14R
7,Plain Strategy,5M Breakout,98,28,70,28.6%,3.6%,14R
8,EMA + Opposite Trade Direction,5M Breakout,38,13,25,34.2%,9.2%,14R
9,With 30M Trend,5M Breakout,60,18,42,30.0%,5.0%,12R


In [480]:
# Display only profitable strategies
display_profitable_strategies(strategy_results)

KeyError: "None of [Index(['Plain Strategy[1M CC]'], dtype='object')] are in the [columns]"

KeyError: "None of [Index(['EMA + Opposite Trade Direction[1M CC]'], dtype='object')] are in the [columns]"

KeyError: "None of [Index(['BOS[1M CC]'], dtype='object')] are in the [columns]"

KeyError: "None of [Index(['Moderate Risk: SL 3-6 pips[1M CC]'], dtype='object')] are in the [columns]"

KeyError: "None of [Index(['Aggressive: SL >= 7 pips[1M CC]'], dtype='object')] are in the [columns]"

...

KeyError: "None of [Index(['With 30M Trend[1M CC]'], dtype='object')] are in the [columns]"

KeyError: "None of [Index(['No News Events[1M CC]'], dtype='object')] are in the [columns]"

KeyError: "None of [Index(['News in 2+ Hours[1M CC]'], dtype='object')] are in the [columns]"

KeyError: "None of [Index(['Aggressive: SL >= 7 pips[5M CC]'], dtype='object')] are in the [columns]"

KeyError: "None of [Index(['Aggressive: SL >= 7 pips[5M Breakout]'], dtype='object')] are in the [columns]"